# Plan 003.3a — Multi-tag content search

A public, synthetic walkthrough of Archiver's bounded multi-tag content search.

Plan 003.3a answers questions such as:

- Which content has **all** of these active tags?
- Which content has **any** of these tags?
- Do the requested tags come from user or system assertions?
- How many current paths refer to each match, without returning an unbounded path or tag list?

The notebook uses only disposable files and leaves no catalog or source data behind.

## Audience, prerequisites, and learning goals

This tutorial is for developers and users who know Archiver's basic catalog and content-tag concepts.

**Prerequisites**

- Run the notebook from the repository root in the project environment.
- Plan 003.2 concepts: content identity, active assertions, provenance, and retraction.

**By the end, you will be able to**

1. build a bounded `all` or `any` multi-tag query;
2. interpret complete content/path totals versus bounded previews;
3. apply provenance without confusing filtering with displayed tags;
4. use the equivalent `archiver catalog tags find` CLI directly;
5. find tagged content after its current paths disappear.

## The semantic boundary

Tags describe SHA-256 **content identity**, not pathnames. Duplicate paths therefore appear once in content results, while their complete current-path count remains visible.

Matching content is catalog-wide. Path counts and previews are scoped to the requested root and its latest successful scan. Retracted assertions never match.

Every numeric preview is bounded in SQLite. Complete totals still describe all matches. Passing `tag_limit=None` is the explicit all-tags mode, and it applies only after parent content rows have been bounded.

## 1. Create disposable synthetic content

Two paths contain identical bytes. A third image and a text note have different content identities. The temporary root is also used by the CLI examples.

In [ ]:
from __future__ import annotations

from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory

from archiver import Catalog, MultiTagContentSearch, TagProvenance

workspace = TemporaryDirectory(prefix="archiver-plan-0033a-")
base = Path(workspace.name)
root = base / "demo-root"
(root / "photos").mkdir(parents=True)
(root / "backups").mkdir()
(root / "notes").mkdir()

(root / "photos" / "day-01.jpg").write_bytes(b"same synthetic photo")
(root / "backups" / "day-01-copy.jpg").write_bytes(b"same synthetic photo")
(root / "photos" / "group.jpg").write_bytes(b"different synthetic photo")
(root / "notes" / "readme.txt").write_text("synthetic notes", encoding="utf-8")

control_directory = root / ".archiver"
control_directory.mkdir()
catalog_path = control_directory / "catalog.sqlite"
Catalog.create(catalog_path).close()

print(f"Disposable root: {root}")
for path in sorted(root.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(root).as_posix())

## 2. Scan with the CLI

The following cell invokes the installed command directly. The Python setup created the disposable catalog at `.archiver/catalog.sqlite`; the CLI excludes that control directory from scans. Refreshing is safe to rerun.

In [ ]:
!archiver catalog scan "{root}" --no-progress

## 3. Add synthetic user and system assertions

Plan 003.3a is query-only; Plan 003.2 supplies the tag mutation API. Here we seed explicit provenance so the later filters are meaningful. The `demo-classifier` is illustrative—Plan 003.3a does not add automatic classification.

In [ ]:
catalog = Catalog.open(catalog_path)

user = TagProvenance("user", "notebook", "1.0")
system = TagProvenance("system", "demo-classifier", "1.0", "synthetic-rules")

shared_photo = catalog.content_for_path(root, PurePosixPath("photos/day-01.jpg"))
group_photo = catalog.content_for_path(root, PurePosixPath("photos/group.jpg"))
notes = catalog.content_for_path(root, PurePosixPath("notes/readme.txt"))

for tag in ("family", "favorite", "trip:us"):
    catalog.add_content_tag(shared_photo, tag, user)
catalog.add_content_tag(shared_photo, "family", system)
catalog.add_content_tag(shared_photo, "picture", system)

for tag in ("family", "trip:us"):
    catalog.add_content_tag(group_photo, tag, user)
catalog.add_content_tag(group_photo, "picture", system)

catalog.add_content_tag(notes, "favorite", user)
catalog.add_content_tag(notes, "text", system)

print("Shared digest:", shared_photo.digest)
print(
    "Duplicate current paths:", [item.relative_path.as_posix() for item in catalog.find_by_content(root, shared_photo)]
)

In [ ]:
# Example of showing the tags for a given path
!archiver catalog tags list "{root}" --path backups/day-01-copy.jpg

## 4. Match all tags or any tag

`match="all"` is the default. Requested names are deduplicated before querying, so repeating `family` does not alter counts. Content rows are ordered by the full digest.

The helper below keeps notebook output compact while showing complete counts and bounded previews.

In [ ]:
def show_search(label: str, result: MultiTagContentSearch) -> None:
    content_label = "content identity" if result.total_matches == 1 else "content identities"
    path_label = "current path" if result.total_current_paths == 1 else "current paths"
    print(f"{label}: {result.total_matches} {content_label}, {result.total_current_paths} {path_label}")
    for item in result.contents:
        overflow = item.active_tag_count - len(item.tags)
        suffix = f" +{overflow}" if overflow else ""
        paths = [path.as_posix() for path in item.current_paths]
        print(
            f"  {item.content_id.digest[:12]}… size={item.size_bytes} "
            f"paths={item.current_path_count} preview={paths} "
            f"tags={list(item.tags)}{suffix}"
        )


all_result = catalog.search_content_by_tags(
    root,
    ("family", "trip:us", "family"),
    match="all",
    limit=20,
    path_limit=20,
    tag_limit=5,
)
any_result = catalog.search_content_by_tags(
    root,
    ("trip:us", "text"),
    match="any",
    limit=20,
    path_limit=20,
    tag_limit=5,
)

show_search("family AND trip:us", all_result)
show_search("trip:us OR text", any_result)

assert all_result.total_matches == 2
assert all_result.total_current_paths == 3
assert any_result.total_matches == 3
assert any_result.total_current_paths == 4

## 5. Separate complete totals from bounded previews

The next query returns only one content row, one path preview, and two tag names. Its totals and per-row counts remain complete. This is the central bounded-result contract.

In [ ]:
bounded = catalog.search_content_by_tags(
    root,
    ("family", "trip:us"),
    limit=1,
    path_limit=1,
    tag_limit=2,
)
show_search("bounded projection", bounded)

assert bounded.total_matches == 2
assert bounded.total_current_paths == 3
assert len(bounded.contents) == 1
assert len(bounded.contents[0].current_paths) <= 1
assert len(bounded.contents[0].tags) <= 2
assert bounded.contents[0].current_path_count >= len(bounded.contents[0].current_paths)
assert bounded.contents[0].active_tag_count >= len(bounded.contents[0].tags)

## 6. Provenance filters matching, not display

A provenance filter must satisfy **every requested tag** under `all` matching. The shared photo has system assertions for both `family` and `picture`; the group photo has only a system `picture` assertion and a user `family` assertion.

Displayed tag previews still contain all active tag names on matched content, regardless of which provenance made the match succeed.

In [ ]:
unfiltered = catalog.search_content_by_tags(root, ("family", "picture"), tag_limit=None)
system_only = catalog.search_content_by_tags(
    root,
    ("family", "picture"),
    provenance="system",
    tag_limit=None,
)
user_only = catalog.search_content_by_tags(
    root,
    ("family", "picture"),
    provenance="user",
    tag_limit=None,
)

show_search("all provenance", unfiltered)
show_search("system provenance", system_only)
show_search("user provenance", user_only)

assert unfiltered.total_matches == 2
assert system_only.total_matches == 1
assert system_only.contents[0].content_id == shared_photo
assert "favorite" in system_only.contents[0].tags  # a user tag is still displayed
assert user_only.total_matches == 0

## 7. Use the multi-tag CLI directly

The CLI uses AND matching by default. It reports complete content/path totals, shortens digests only for display, and can show bounded path details and tag previews.

In [ ]:
!archiver catalog tags find "{root}" family trip:us --details --path-limit 2 --display-tag-limit 3

`--match any` changes only the requested-tag expression. `--limit` bounds content rows while the summary remains complete.

In [ ]:
!archiver catalog tags find "{root}" trip:us text --match any --limit 2

`--provenance system` applies to every requested tag. `--all-tags` displays every active tag only for already bounded content rows and wraps when needed.

In [ ]:
!archiver catalog tags find "{root}" family picture --provenance system --all-tags --details --path-limit 1

## 8. Tagged content survives path disappearance

The notebook now removes the two **synthetic** duplicate files and refreshes the disposable catalog. Archiver's scan remains observational: the notebook performs the deletions explicitly.

The shared content identity remains in tag search, but its root-scoped current-path count becomes zero.

In [ ]:
(root / "photos" / "day-01.jpg").unlink()
(root / "backups" / "day-01-copy.jpg").unlink()
print("Synthetic duplicate paths removed by the notebook.")

In [ ]:
!archiver catalog scan "{root}" --no-progress

In [ ]:
after_disappearance = catalog.search_content_by_tags(
    root,
    ("family", "trip:us"),
    path_limit=20,
    tag_limit=None,
)
show_search("after path disappearance", after_disappearance)

shared_row = next(item for item in after_disappearance.contents if item.content_id == shared_photo)
assert shared_row.current_path_count == 0
assert shared_row.current_paths == ()
assert after_disappearance.total_matches == 2
assert after_disappearance.total_current_paths == 1

In [ ]:
!archiver catalog tags find "{root}" family trip:us --details --all-tags

## Exercise

Predict the result of an `all` search for `family` and `picture` with `provenance="system"` after the duplicate paths disappeared.

- How many content identities match?
- How many current paths do those matches have?
- Will user tags such as `favorite` still appear in the returned tag list?

Try writing the query before revealing the answer cell.

In [ ]:
# Answer
exercise = catalog.search_content_by_tags(
    root,
    ("family", "picture"),
    provenance="system",
    tag_limit=None,
)
show_search("exercise answer", exercise)

assert exercise.total_matches == 1
assert exercise.total_current_paths == 0
assert exercise.contents[0].content_id == shared_photo
assert "favorite" in exercise.contents[0].tags

## Pitfalls and extensions

- **Do not count duplicate paths as duplicate content results.** One row is one content identity; `current_path_count` reports its current instances at the requested root.
- **Do not treat provenance as a display filter.** It filters how requested tags qualify. Returned tag previews show all active tag names.
- **Do not infer absence from a bounded preview.** Compare `current_path_count` with `len(current_paths)` and `active_tag_count` with `len(tags)`.
- **Do not use a path from another location without its root.** Current paths are explicitly scoped to the root passed to the query.
- **Use `tag_limit=None` deliberately.** It may materialize all tags, but only for already bounded content rows.

Optional extension: add another catalog location containing the same bytes and compare root-scoped path counts while the catalog-wide content match remains unchanged.

## Cleanup

In [ ]:
catalog.close()
workspace.cleanup()
print("Temporary catalog and synthetic files removed.")